# Kiểm thử Khả năng Truy xuất của RAG (Retrieval Testing & K Optimization)

Thực hiện việc kiểm thử khả năng truy xuất của RAG từ Vector Database ChromaDB đã được tạo.   
Chúng ta sẽ nghiên cứu cách tìm kiếm tương đồng và lựa chọn kích thước ngữ cảnh truy xuất $K$ tối ưu cho phần cứng laptop (8GB VRAM).

In [1]:
import os
import sys
from dotenv import load_dotenv

# Nạp các biến môi trường cấu hình cache
load_dotenv(os.path.abspath("../.env"))

import pandas as pd

# Thêm thư mục src vào path để import rag_utils
sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tải Cơ sở dữ liệu Vector

In [2]:
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)
print("✔ Đã tải thành công ChromaDB.")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4314.05it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
t:\5 - Summer 2026\AES_LLM\src\rag\rag_utils.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


✔ Đã tải thành công ChromaDB.


## 2. Thử nghiệm Truy xuất Thực tế

Lấy một bài luận ngẫu nhiên để truy xuất xem ChromaDB có trả về các bài luận có chủ đề và mức điểm tương quan hợp lý không.

In [3]:
# Bài luận mẫu để truy xuất
sample_query_essay = """
Some people think that universities should provide graduates with the knowledge and skills needed in the workplace.
Others think that the true function of a university should be to give access to knowledge for its own sake, regardless of whether the course is useful to an employer.
In my opinion, universities must focus on providing practical skills because finding a job is very important for graduates today.
"""

# Kiểm thử với K = 2 (Đề xuất tối ưu cho 8GB VRAM)
K_VAL = 2
retrieved_docs = rag_utils.retrieve_examples(vectordb, sample_query_essay, k=K_VAL)

print(f"=== ĐÃ TRUY XUẤT ĐƯỢC {len(retrieved_docs)} BÀI VIẾT PHÙ HỢP ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Ví dụ {i+1} ---")
    print(f"Điểm Overall Band: {doc.metadata.get('Overall_Band')}")
    print(f"Điểm các tiêu chí: TR={doc.metadata.get('TR_Band')} | CC={doc.metadata.get('CC_Band')} | LR={doc.metadata.get('LR_Band')} | GRA={doc.metadata.get('GRA_Band')}")
    # In 300 ký tự đầu tiên của bài viết tham khảo
    print("Nội dung bài luận:", doc.page_content[:300] + "...")
    print("-" * 50)

=== ĐÃ TRUY XUẤT ĐƯỢC 2 BÀI VIẾT PHÙ HỢP ===

--- Ví dụ 1 ---
Điểm Overall Band: 7.5
Điểm các tiêu chí: TR=8.0 | CC=8.0 | LR=7.0 | GRA=7.0
Nội dung bài luận: Prompt: Nations should spend more money on skills and vocational training for practical work, rather than on university education. To what extent do you agree or disagree?

Essay: Many today feel that countries should prioritise vocational training over higher education due to changes in the labour ...
--------------------------------------------------
--- Ví dụ 2 ---
Điểm Overall Band: 7.5
Điểm các tiêu chí: TR=8.0 | CC=8.0 | LR=7.0 | GRA=7.0
Nội dung bài luận: Prompt: Nations should spend more money on skills and vocational training for practical work, rather than on university education. To what extent do you agree or disagree?

Essay: Many today feel that countries should prioritise vocational training over higher education due to changes in the labour ...
--------------------------------------------------
